### what is [linker script (.ld)](https://www.youtube.com/watch?v=B7oKdUvRhQQ&list=PLERTijJOmYrDiiWd10iRHY0VRHdJwUH4g&index=4)

- 基本上就是強制設定說  text data bss 的位址要在哪裡   



### storage of final executable in code memory  

```
高位址            
0x40_0000        stack 往下長
                     |
                     v
---------------------------------------

code memory      Unused code memory (unused sram) 
(flash)          
---------------------------------------
                   heap
--------------------------------------
                 .bss (un initialized vaar)
                 .data (initialized global and static var)
                 .rodata  
                 .text  
0x08_0000      vector table      
低位址                   
```

### example 1 a-class_fail_todo  (我覺得這樣寫才是正確的)
#### linker.ld

```
SECTIONS{
    .text.boot : { *(.text.boot) }  //對應到 boot.s  .section .text.boot
    .text : { *(.text) }
    .rodata : { *(.rodata) }
    .data : { *(.data) }
    . = ALIGN(0x8);         //??????
    bss_begin = .;
    .bss : { *(.bss*) }
    bss_end = .;
}
```

#### mm.h  
```
PAGE_SHIFT = 12 → 頁大小 4 KB (1 << 12)
TABLE_SHIFT = 9 → 每層頁表 512 entry
SECTION_SHIFT = PAGE_SHIFT + TABLE_SHIFT = 21 → 每個 section 大小 = 1 << 21 = 2 MB

LOW_MEMORY = 2 * SECTION_SIZE = 4 MB

stack pointer 被設到 0x400000 (4 MB)。
```
#### boot.S

```
.section ".text.boot"
.globl _start
_start:
   mov sp, #LOW_MEMORY 
```

### example2  b_easy_baremetal_led  (我覺得這樣寫怪怪的)
#### linker.ld  
```
ENTRY(_start)  //start_lable in start.s
SECTIONS{
   . = 0x80000;   stack pointer point to here  ???
   .text :{ *(.text*)}  //對應到 start.s  .section .text
   .data :{ *(.data*)}
   .bss :{*(.bss*)}
}

. = 0x80000;  代表 .text 會從 0x80000 開始往上排
stack 往哪裡長？ ARM stack 是 向下長 也就是： push → SP 減少
程式往上, stack 往下,兩邊不會撞在一起。

潛在問題：
你的 stack 起始點 = 程式起始點
但如果你 stack 用太多，有可能壓到 .bss 或 .data
不過你現在程式幾乎沒用 stack，所以沒事。


高位址         

code memory      Unused code memory 
(flash)          .data (initialized global and static var)
                 .rodata  
                 .text  
0x08_0000      vector table      
          stack 往下長
低位址     

和典型的架構不符合  典型架構  stack 應該要再 text 上面
```

#### start.s 

```s
.global _start   
.section .text
_start: 
   ldr x0, =0x80000   設定 stack pointer 
   mov sp, x0  
```